In [26]:
import sys
import os
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm import tqdm

# Set the project root (one level up from /notebooks/)
project_root = Path(os.getcwd()).parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import your classes and functions directly from the source files
from src.dataset.eyes_dataset import build_dataloaders, EyesDataset
from src.models.binary_roi_classifier import build_binary_classifier

In [15]:
STAGE_1_EPOCHS     = 50
STAGE_2_EPOCHS     = 50
BATCH_SIZE         = 64
STAGE_1_LR         = 1e-3
STAGE_2_HEAD_LR    = 1e-3
STAGE_2_BACKBONE_LR = 1e-4
MIXUP_ALPHA        = 0.4
EARLY_STOP_PATIENCE = 50
 
DATASET_ROOTS = {
    "eyes":  "../dataset_split/train/eyes",
    "mouth": "../dataset_split/train/mouth",
}
 
CLASS_LABELS = {
    "eyes":  {"closed": 0, "open": 1},
    "mouth": {"closed": 0, "open": 1},
}
 
CHECKPOINT_DIR = Path("runs/binary")

In [16]:
from pathlib import Path

roi = "eyes"
root_path = Path(DATASET_ROOTS[roi])
print(f"Checking root: {root_path.absolute()}")

for class_name in CLASS_LABELS[roi].keys():
    class_dir = root_path / class_name
    images = list(class_dir.glob("*.png"))
    print(f"Folder '{class_name}': Found {len(images)} .png images")

Checking root: c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset_split\train\eyes
Folder 'closed': Found 2296 .png images
Folder 'open': Found 21042 .png images


In [17]:
# =========================
# Sanity check
# =========================
def check_dataset(roi):
    root_path = Path(DATASET_ROOTS[roi])
    print(f"Checking root: {root_path.absolute()}")
    for class_name in CLASS_LABELS[roi].keys():
        class_dir = root_path / class_name
        images = list(class_dir.glob("*.png"))
        print(f"  Folder '{class_name}': Found {len(images)} .png images")

In [18]:
def mixup_batch(images, labels, alpha):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size = images.size(0)
    perm = torch.randperm(batch_size, device=images.device)
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam

def mixup_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss  = total_loss / len(loader.dataset)
    acc       = accuracy_score(all_labels, all_preds)
    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, precision, recall, f1

In [19]:
def run_stage(
    model, optimizer, criterion,
    train_loader, val_loader,
    device, epochs, start_epoch,
    use_mixup, checkpoint_path,
):
    best_val_f1 = 0.0
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(start_epoch, start_epoch + epochs):
        model.train()
        running_loss   = 0.0
        correct        = 0
        total          = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup:
                mixed, la, lb, lam = mixup_batch(images, labels, MIXUP_ALPHA)
                logits = model(mixed)
                loss = mixup_loss(criterion, logits, la, lb, lam)
            else:
                logits = model(images)
                loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct      += (logits.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc  = correct / total

        val_loss, val_acc, prec, rec, f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(f1)

        print(
            f"Epoch {epoch:>3} | "
            f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f} | "
            f"prec={prec:.3f}  rec={rec:.3f}  F1={f1:.3f}"
        )

        if f1 > best_val_f1:
            best_val_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> Saved best model (F1={f1:.3f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print("Early stop triggered.")
                break

    return history

In [23]:
def plot_history(history: dict, roi: str, save_dir: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Training History — {roi.upper()}", fontsize=13)

    # --- Loss ---
    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="steelblue")
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   color="tomato")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # --- Accuracy ---
    axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="steelblue")
    axes[1].plot(epochs, history["val_acc"],   label="Val Acc",   color="tomato")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = save_dir / "training_history.png"
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved training history -> {save_path}")

In [20]:
def build_sampler(all_samples, train_indices, num_classes):
    """Build a WeightedRandomSampler from the training split only."""
    targets = torch.tensor([all_samples[i][1] for i in train_indices])
    class_counts = torch.bincount(targets, minlength=num_classes).float()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]
 
    print(f"  Class counts  : {class_counts.tolist()}")
    print(f"  Class weights : {[f'{w:.4f}' for w in class_weights.tolist()]}")
 
    return (
        WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        ),
        class_weights,
    )

In [ ]:
def train_roi(roi):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*50}")
    print(f"Training ROI: {roi.upper()} on {device}")
    print(f"{'='*50}")

    check_dataset(roi)

    root        = Path(DATASET_ROOTS[roi])
    num_classes = len(CLASS_LABELS[roi])

    # --- 1. Build dataloaders with downsampling (no sampler needed) ---
    train_loader, val_loader, train_indices = build_dataloaders(
        root=root,
        class_to_label=CLASS_LABELS[roi],
        batch_size=BATCH_SIZE,
        sampler=None,
        downsample_majority=True,   # majority capped to match minority
    )

    # --- 2. Compute class weights from downsampled train split ---
    all_samples = train_loader.dataset.samples
    targets = torch.tensor([label for _, label in all_samples])
    class_counts  = torch.bincount(targets, minlength=num_classes).float()
    class_weights = 1.0 / class_counts
    print(f"  Class counts  : {class_counts.tolist()}")
    print(f"  Class weights : {[f'{w:.4f}' for w in class_weights.tolist()]}")

    # --- 3. Weighted loss ---
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    # --- 4. Model ---
    model = build_binary_classifier(pretrained=True, freeze_backbone=True).to(device)
    checkpoint_dir  = CHECKPOINT_DIR / roi
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "best.pt"

    # --- Stage 1: Frozen backbone ---
    print("\n--- Stage 1: Frozen Backbone ---")
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=STAGE_1_LR,
    )
    history_s1 = run_stage(
        model, optimizer, criterion,
        train_loader, val_loader, device,
        epochs=STAGE_1_EPOCHS, start_epoch=1,
        use_mixup=False,
        checkpoint_path=checkpoint_path,
    )

    # --- Stage 2: Fine-tune ---
    print("\n--- Stage 2: Fine-tuning ---")
    model.freeze_backbone(toggle=False, last_n_blocks=2)
    optimizer = torch.optim.Adam([
        {"params": model.backbone.parameters(), "lr": STAGE_2_BACKBONE_LR},
        {"params": model.head.parameters(),     "lr": STAGE_2_HEAD_LR},
    ])
    history_s2 = run_stage(
        model, optimizer, criterion,
        train_loader, val_loader, device,
        epochs=STAGE_2_EPOCHS, start_epoch=STAGE_1_EPOCHS + 1,
        use_mixup=True,
        checkpoint_path=checkpoint_path,
    )

    print(f"\nDone. Best model saved to: {checkpoint_path}")

    full_history = {
        key: history_s1[key] + history_s2[key]
        for key in history_s1
    }
    results_dir = Path("runs/eval") / roi
    results_dir.mkdir(parents=True, exist_ok=True)
    plot_history(full_history, roi, results_dir)

In [29]:
train_roi("mouth")


Training ROI: MOUTH on cuda
Checking root: c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset_split\train\mouth
  Folder 'closed': Found 11607 .png images
  Folder 'open': Found 213 .png images
  Downsampling all classes to 213 samples
  Class counts  : [182.0, 182.0]
  Class weights : ['0.0055', '0.0055']

--- Stage 1: Frozen Backbone ---


Epoch   1 | train_loss=0.6100  train_acc=0.624 | val_loss=0.4980  val_acc=0.790 | prec=0.852  rec=0.790  F1=0.781
  -> Saved best model (F1=0.781)


Epoch   2 | train_loss=0.3973  train_acc=0.857 | val_loss=0.3778  val_acc=0.903 | prec=0.919  rec=0.903  F1=0.902
  -> Saved best model (F1=0.902)


Epoch   3 | train_loss=0.3062  train_acc=0.865 | val_loss=0.3417  val_acc=0.855 | prec=0.887  rec=0.855  F1=0.852


Epoch   4 | train_loss=0.2405  train_acc=0.898 | val_loss=0.3692  val_acc=0.823 | prec=0.869  rec=0.823  F1=0.817


...

Epoch  97 | train_loss=0.3117  train_acc=0.668 | val_loss=0.1349  val_acc=0.984 | prec=0.984  rec=0.984  F1=0.984


Epoch  98 | train_loss=0.2605  train_acc=0.835 | val_loss=0.1231  val_acc=0.984 | prec=0.984  rec=0.984  F1=0.984


Epoch  99 | train_loss=0.2967  train_acc=0.824 | val_loss=0.1341  val_acc=0.984 | prec=0.984  rec=0.984  F1=0.984


Epoch 100 | train_loss=0.2583  train_acc=0.692 | val_loss=0.1342  val_acc=0.984 | prec=0.984  rec=0.984  F1=0.984

Done. Best model saved to: runs\binary\mouth\best_1.pt


TypeError: can only concatenate list (not "dict") to list

In [ ]:
from torchview import draw_graph

model = build_binary_classifier(
    pretrained=False,
    freeze_backbone=True
)

# Only visualize classifier head
head = model.head
head.eval()

# Input to head = pooled backbone output
dummy_input = torch.randn(1, 576)

graph = draw_graph(
    head,
    input_data=dummy_input,
    expand_nested=True,
    save_graph=True,
    filename="cnn_head_architecture",
    directory="./diagrams"
)

graph.visual_graph.graph_attr["rankdir"] = "LR"

graph.visual_graph.render(format="png")